In [ ]:
# Load R magic extension for Python Jupyter kernel (Kaggle / Colab support)
try:
    %load_ext rpy2.ipython
except Exception as e:
    print("Note on rpy2 initialization:", e)

# Hierarchical Triage Pipeline Benchmark (`models/combined_hierarchical_triage_pipeline.ipynb`)

This notebook evaluates the **5-Class Probabilistic Triage Pipeline** (Layer 2 Grouped Random Forest + Layer 3 Dual LightGBM Specialists) on the **Holdout Test Set** (ratios parsed strictly from `config/triage_conf.json`), benchmarked on **Recall**, **Specificity**, **Balanced Accuracy**, and **ROC-AUC**:

### System Architecture & Workflow
1. **Predictor Feature Space (13 Total Features)**:
   - **Raw Demographics & Chief Complaint**: `age`, `gender`, `cc_breathingdifficulty`.
   - **10 Binary Vital Anomaly Flags**: `is_dyspnea_total`, `is_dyspnea_moderate`, `is_bradypnea`, `is_tachypnea`, `is_hypotension`, `is_hypertension`, `is_bradycardia_total`, `is_bradycardia_moderate`, `is_tachycardia_total`, `is_tachycardia_moderate`.
2. **Layer 2 (Random Forest 3-Class Grouped Model)**:
   - Predicts probabilities for classes `"1"` ($P_2(\text{ESI 1})$), `"2_3"` ($P_2(\text{ESI 2/3})$), and `"4_5"` ($P_2(\text{ESI 4/5})$).
3. **Layer 3 (Dual LightGBM Specialist Ensemble)**:
   - LightGBM 1 predicts $P(\text{ESI 2} \mid \text{ESI 2/3})$ and $P(\text{ESI 3} \mid \text{ESI 2/3})$.
   - LightGBM 2 predicts $P(\text{ESI 4} \mid \text{ESI 4/5})$ and $P(\text{ESI 5} \mid \text{ESI 4/5})$.
4. **5-Class Joint Probability Calculation**:
   - $P(\text{ESI 1}) = P_2(\text{ESI 1})$
   - $P(\text{ESI 2}) = P_2(\text{ESI 2/3}) \times P(\text{ESI 2} \mid \text{ESI 2/3})$
   - $P(\text{ESI 3}) = P_2(\text{ESI 2/3}) \times P(\text{ESI 3} \mid \text{ESI 2/3})$
   - $P(\text{ESI 4}) = P_2(\text{ESI 4/5}) \times P(\text{ESI 4} \mid \text{ESI 4/5})$
   - $P(\text{ESI 5}) = P_2(\text{ESI 4/5}) \times P(\text{ESI 5} \mid \text{ESI 4/5})$

### Target Benchmark Metrics Suite
Evaluates ONLY **Recall (Sensitivity)**, **Specificity**, **Balanced Accuracy**, and **ROC-AUC**.

In [ ]:
%%R
# ---------------------------------------------------------
# Step 1: Load Required Libraries & Parse Configuration JSON
# ---------------------------------------------------------
suppressPackageStartupMessages({
  library(jsonlite)
  library(caret)
  library(dplyr)
  library(ggplot2)
  library(tidyr)
  library(pROC)
  library(xgboost)
  library(ranger)
})
has_lgb <- requireNamespace("lightgbm", quietly = TRUE)
if (has_lgb) library(lightgbm)
config_path <- "../config/triage_conf.json"
if (!file.exists(config_path)) {
  config_path <- "config/triage_conf.json"
}
config <- fromJSON(config_path)
cat("=== Master Hierarchical Triage Pipeline Initialized ===\n")
cat("Data Source Path:", config$path$data_source, "\n")
cat("Target Column:   ", config$classes$target_col, "\n")
cat("Test Size:       ", config$training$test_size, "\n")
cat("Random State:    ", config$training$random_state, "\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 2: Load Full Dataset & Construct 13 Predictor Features
# ---------------------------------------------------------
set.seed(config$training$random_state)
data_file <- config$path$data_source
if (!file.exists(data_file) && file.exists(paste0("../", data_file))) {
  data_file <- paste0("../", data_file)
}
cat("Loading dataset from:", data_file, "...\n")
data_env <- new.env()
load(data_file, envir = data_env)
df_names <- ls(data_env)[sapply(ls(data_env), function(x) is.data.frame(get(x, envir = data_env)))]
df_sizes <- sapply(df_names, function(x) nrow(get(x, envir = data_env)))
data_obj_name <- df_names[which.max(df_sizes)]
raw_df <- get(data_obj_name, envir = data_env)
target_col_name <- config$classes$target_col
gender_vec <- if ("gender" %in% names(raw_df)) ifelse(as.character(raw_df$gender) == "Male", 1, 0) else 0
cc_bd_vec  <- if ("cc_breathingdifficulty" %in% names(raw_df)) ifelse(!is.na(raw_df$cc_breathingdifficulty), raw_df$cc_breathingdifficulty, 0) else 0
# Construct 13 Predictor Features
df_full <- data.frame(
  age                     = raw_df$age,
  gender                  = gender_vec,
  cc_breathingdifficulty  = cc_bd_vec,
  
  # 10 Binary Vital Anomaly Flags
  is_dyspnea_total        = ifelse(!is.na(raw_df$triage_vital_o2) & raw_df$triage_vital_o2 < 90, 1, 0),
  is_dyspnea_moderate     = ifelse(!is.na(raw_df$triage_vital_o2) & raw_df$triage_vital_o2 >= 90 & raw_df$triage_vital_o2 < 94, 1, 0),
  is_bradypnea            = ifelse(!is.na(raw_df$triage_vital_rr) & raw_df$triage_vital_rr < 10, 1, 0),
  is_tachypnea            = ifelse(!is.na(raw_df$triage_vital_rr) & raw_df$triage_vital_rr > 30, 1, 0),
  is_hypotension          = ifelse(!is.na(raw_df$triage_vital_sbp) & raw_df$triage_vital_sbp <= 90, 1, 0),
  is_hypertension         = ifelse(!is.na(raw_df$triage_vital_sbp) & raw_df$triage_vital_sbp > 220, 1, 0),
  is_bradycardia_total    = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr < 40, 1, 0),
  is_bradycardia_moderate = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr >= 40 & raw_df$triage_vital_hr < 60, 1, 0),
  is_tachycardia_total    = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr > 150, 1, 0),
  is_tachycardia_moderate = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr > 100 & raw_df$triage_vital_hr <= 150, 1, 0)
)
raw_esi <- as.character(raw_df[[target_col_name]])
df_full$target_col <- factor(raw_esi, levels = c("1", "2", "3", "4", "5"))
initial_rows <- nrow(df_full)
df_full <- na.omit(df_full)
cat(sprintf("Complete Case Filtering: Removed %d rows (Remaining: %d)\n", initial_rows - nrow(df_full), nrow(df_full)))
# Stratified Test Partitioning (Parsed from config/triage_conf.json)
test_size <- config$training$test_size
in_train_val <- createDataPartition(df_full$target_col, p = 1 - test_size, list = FALSE)
test_df      <- df_full[-in_train_val, ]
cat(sprintf("Holdout Test Set Ready (from config test_size=%.4f): %d rows x %d cols\n", test_size, nrow(test_df), ncol(test_df)))
cat("Natural 5-Class Target Distribution (ESI 1 to 5):\n")
print(table(test_df$target_col))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 3: Load Saved Model Artifacts from deploy/
# ---------------------------------------------------------
deploy_dir <- "../deploy"
if (!dir.exists(deploy_dir)) deploy_dir <- "deploy"
path_rf    <- file.path(deploy_dir, "rf_esi23_esi45_extreme_model.rds")
path_ds    <- file.path(deploy_dir, "xgboost_esi23_esi45_extreme_model.rds")
path_lgb23 <- file.path(deploy_dir, "lightgbm_esi23_model.rds")
path_lgb45 <- file.path(deploy_dir, "lightgbm_esi45_model.rds")
cat("Loading model artifacts...\n")
art_rf    <- readRDS(path_rf)
art_ds    <- readRDS(path_ds)
art_lgb23 <- if (file.exists(path_lgb23)) readRDS(path_lgb23) else NULL
art_lgb45 <- if (file.exists(path_lgb45)) readRDS(path_lgb45) else NULL
cat("  - Layer 2 Model (Random Forest 3-Class Grouped Model) Loaded.\n")
cat("  - Layer 3A Model (LightGBM ESI 2 vs 3 Specialist) Loaded.\n")
cat("  - Layer 3B Model (LightGBM ESI 4 vs 5 Specialist) Loaded.\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 4: Run 5-Class Hierarchical Pipeline Inference
# ---------------------------------------------------------
# 1. Layer 2: Random Forest 3-Class Grouped Model ('2_3', '4_5', '1') (13 Features)
rf_train_feats <- art_rf$model$independent.variable.names
if (is.null(rf_train_feats)) rf_train_feats <- art_rf$model$forest$independent.variable.names
if (is.null(rf_train_feats)) rf_train_feats <- setdiff(names(test_df), "target_col")
cont_cols_rf   <- intersect(rf_train_feats, names(art_rf$preproc$mean))
binary_cols_rf <- setdiff(rf_train_feats, cont_cols_rf)
scaled_cont_rf <- predict(art_rf$preproc, test_df[, cont_cols_rf, drop = FALSE])
test_rf_scaled <- cbind(scaled_cont_rf, test_df[, binary_cols_rf, drop = FALSE])
test_rf_scaled <- test_rf_scaled[, rf_train_feats, drop = FALSE]
rf_raw_probs <- predict(art_rf$model, data = test_rf_scaled)$predictions
p_rf_23 <- rf_raw_probs[, "2_3"]
p_rf_45 <- rf_raw_probs[, "4_5"]
p_rf_1  <- if ("1" %in% colnames(rf_raw_probs)) rf_raw_probs[, "1"] else rf_raw_probs[, "other"]
# 2. Layer 3: Specialist Conditional Probabilities (13 Features)
ds_train_feats <- names(art_ds$preproc$mean)
ds_train_feats <- union(ds_train_feats, setdiff(names(test_df), c(ds_train_feats, "target_col")))
ds_train_feats <- intersect(names(test_df), ds_train_feats)
cont_cols_ds   <- intersect(ds_train_feats, names(art_ds$preproc$mean))
binary_cols_ds <- setdiff(ds_train_feats, cont_cols_ds)
scaled_cont_ds <- predict(art_ds$preproc, test_df[, cont_cols_ds, drop = FALSE])
test_ds_scaled <- cbind(scaled_cont_ds, test_df[, binary_cols_ds, drop = FALSE])
test_ds_x      <- as.matrix(test_ds_scaled[, ds_train_feats, drop = FALSE])
# Layer 3A (ESI 2 vs 3 LightGBM)
if (!is.null(art_lgb23) && art_lgb23$has_lgb) {
  p_esi2_given_23 <- predict(art_lgb23$model, test_ds_x)
} else if (!is.null(art_ds$model_lgb23)) {
  p_esi2_given_23 <- predict(art_ds$model_lgb23, test_ds_x)
} else if (!is.null(art_ds$model_lgb)) {
  p_esi2_given_23 <- predict(art_ds$model_lgb, test_ds_x)
} else {
  p_esi2_given_23 <- predict(art_ds$model_lgb, xgb.DMatrix(data = test_ds_x))
}
p_esi3_given_23 <- 1 - p_esi2_given_23
# Layer 3B (ESI 4 vs 5 LightGBM)
if (!is.null(art_lgb45) && art_lgb45$has_lgb) {
  p_esi4_given_45 <- predict(art_lgb45$model, test_ds_x)
} else if (!is.null(art_ds$model_lgb45)) {
  p_esi4_given_45 <- predict(art_ds$model_lgb45, test_ds_x)
} else if (!is.null(art_ds$model_xgb)) {
  p_esi4_given_45 <- predict(art_ds$model_xgb, test_ds_x)
} else {
  p_esi4_given_45 <- predict(art_ds$model_xgb, xgb.DMatrix(data = test_ds_x))
}
p_esi5_given_45 <- 1 - p_esi4_given_45
# ---------------------------------------------------------
# 5-CLASS JOINT PROBABILITY CALCULATION
# ---------------------------------------------------------
p_esi1 <- p_rf_1  # DIRECT LAYER 2 PREDICTION FOR ESI 1
p_esi2 <- p_rf_23 * p_esi2_given_23
p_esi3 <- p_rf_23 * p_esi3_given_23
p_esi4 <- p_rf_45 * p_esi4_given_45
p_esi5 <- p_rf_45 * p_esi5_given_45
probs_pipeline <- cbind(p_esi1, p_esi2, p_esi3, p_esi4, p_esi5)
colnames(probs_pipeline) <- c("1", "2", "3", "4", "5")
cat(sprintf("Probability Sum Check: Mean = %.6f (Range: %.4f - %.4f)\n",
            mean(rowSums(probs_pipeline)), min(rowSums(probs_pipeline)), max(rowSums(probs_pipeline))))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 5: Benchmark Pipeline Performance (Recall, Specificity, BalAcc, ROC-AUC ONLY)
# ---------------------------------------------------------
act_fac <- factor(test_df$target_col, levels = c("1", "2", "3", "4", "5"))
pred_idx <- apply(probs_pipeline, 1, which.max)
pred_fac <- factor(colnames(probs_pipeline)[pred_idx], levels = c("1", "2", "3", "4", "5"))
cm  <- confusionMatrix(pred_fac, act_fac)
rec_by_cls     <- as.numeric(cm$byClass[, "Sensitivity"])
spec_by_cls    <- as.numeric(cm$byClass[, "Specificity"])
bal_acc_by_cls <- as.numeric(cm$byClass[, "Balanced Accuracy"])
rec_by_cls[is.na(rec_by_cls)]         <- 0
spec_by_cls[is.na(spec_by_cls)]       <- 0
bal_acc_by_cls[is.na(bal_acc_by_cls)] <- 0
roc_auc_by_cls <- sapply(1:5, function(i) {
  cls_name <- levels(act_fac)[i]
  act_bin  <- ifelse(act_fac == cls_name, 1, 0)
  r_obj    <- tryCatch(pROC::roc(act_bin, probs_pipeline[, i]), error = function(e) NULL)
  if (!is.null(r_obj)) as.numeric(r_obj$auc) else NA
})
macro_rec     <- mean(rec_by_cls)
macro_spec    <- mean(spec_by_cls)
macro_bal_acc <- mean(bal_acc_by_cls)
macro_auc     <- mean(roc_auc_by_cls, na.rm = TRUE)
cat(sprintf("============================================================\n"))
cat(sprintf("   5-CLASS HIERARCHICAL TRIAGE PIPELINE TEST BENCHMARK\n"))
cat(sprintf("============================================================\n"))
cat(sprintf("  Macro Recall (Sens)     : %.4f\n", macro_rec))
cat(sprintf("  Macro Specificity       : %.4f\n", macro_spec))
cat(sprintf("  Macro Balanced Accuracy : %.4f\n", macro_bal_acc))
cat(sprintf("  Macro ROC-AUC           : %.4f\n", macro_auc))
cat(sprintf("============================================================\n\n"))
print(cm$table)
# Write CSV Summary Report
reports_dir <- "../reports"
if (!dir.exists(reports_dir)) reports_dir <- "reports"
if (!dir.exists(reports_dir)) dir.create(reports_dir, recursive = TRUE)
pipeline_report_df <- data.frame(
  Pipeline_Model          = "Hierarchical_Triage_Pipeline",
  Macro_Recall            = round(macro_rec, 4),
  Macro_Specificity       = round(macro_spec, 4),
  Macro_Balanced_Accuracy = round(macro_bal_acc, 4),
  Macro_ROC_AUC           = round(macro_auc, 4)
)
write.csv(pipeline_report_df, file = file.path(reports_dir, "combined_pipeline_test_report.csv"), row.names = FALSE)
cat("\nHierarchical Pipeline Test Report written to: reports/combined_pipeline_test_report.csv\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 6: Diagnostic Plots (Metrics Bar Chart for Recall, Specificity, BalAcc, ROC-AUC ONLY)
# ---------------------------------------------------------
plots_dir <- "../plots"
if (!dir.exists(plots_dir)) plots_dir <- "plots"
if (!dir.exists(plots_dir)) dir.create(plots_dir, recursive = TRUE)
metrics_df <- data.frame(
  Metric = c("Recall", "Specificity", "Balanced_Acc", "ROC_AUC"),
  Score  = c(macro_rec, macro_spec, macro_bal_acc, macro_auc)
)
p_bar <- ggplot(metrics_df, aes(x = Metric, y = Score, fill = Metric)) +
  geom_bar(stat = "identity", width = 0.5) +
  geom_text(aes(label = sprintf("%.3f", Score)), vjust = -0.3, size = 3.5, fontface = "bold") +
  theme_minimal() +
  scale_fill_manual(values = c("Recall" = "#2b5c8f", "Specificity" = "#e07a5f", "Balanced_Acc" = "#3d405b", "ROC_AUC" = "#81b29a")) +
  labs(title = "Hierarchical Triage Pipeline Holdout Test Performance",
       subtitle = "Macro Performance Scores (13 Predictor Features)",
       y = "Score", x = "") +
  theme(plot.title = element_text(face = "bold", size = 13, hjust = 0.5),
        plot.subtitle = element_text(size = 9.5, hjust = 0.5),
        legend.position = "none")
ggsave(file.path(plots_dir, "combined_pipeline_metrics_barchart.png"), plot = p_bar, width = 8.0, height = 4.8, dpi = 300)
cat("Pipeline Bar Chart saved to: plots/combined_pipeline_metrics_barchart.png\n")
p_bar